In [ ]:
!nvidia-smi

Fri Oct 30 10:39:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   41C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch, sys
print("Python :", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("GPU OK :", torch.cuda.is_available())
!nvcc --version | tail -2

Python : 3.13.16
PyTorch: 2.11.0+cu130
Torch CUDA: 13.0
GPU OK : True
Cuda compilation tools, release 13.0, V13.0.88
Build cuda_13.0.r13.0/compiler.36424714_0


In [ ]:
%cd /content
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting
%cd /content/gaussian-splatting
!ls
!ls submodules

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 1053, done.
remote: Total 1053 (delta 0), reused 0 (delta 0), pack-reused 1053 (from 1)
Receiving objects: 100% (1053/1053), 78.72 MiB | 19.99 MiB/s, done.
Resolving deltas: 100% (593/593), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects:

In [ ]:
!pip install -q plyfile tqdm opencv-python joblib

In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "8.9"
%cd /content/gaussian-splatting/submodules
f = "simple-knn/simple_knn.cu"
s = open(f).read()
if "<float.h>" not in s:
    open(f, "w").write("#include <float.h>\n" + s); print("patched", f)
f = "diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h"
s = open(f).read()
if "<cstdint>" not in s:
    open(f, "w").write("#include <cstdint>\n" + s); print("patched", f)

/content/gaussian-splatting/submodules
patched simple-knn/simple_knn.cu
patched diff-gaussian-rasterization/cuda_rasterizer/rasterizer_impl.h


In [ ]:
!pip install --no-build-isolation ./simple-knn 2>&1 | tail -5

  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp313-cp313-linux_x86_64.whl size=3210808 sha256=7bb84e057901c0d3a67310e540d06fa2b4378516b62e2abcc9d0d96ffb91b417
  Stored in directory: /tmp/pip-ephem-wheel-cache-jags4qb8/wheels/9c/6f/a6/86f8f308d51a82157b32336055b058903d71f74c2ca4e9cd85
Successfully built simple_knn


In [ ]:
!pip install --no-build-isolation ./diff-gaussian-rasterization 2>&1 | tail -5

  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp313-cp313-linux_x86_64.whl size=3360126 sha256=a13d2176846ee23b8cc8e6bcf0a2c2427953bd8326559411ed29ece93d4024f4
  Stored in directory: /tmp/pip-ephem-wheel-cache-te0o2en2/wheels/49/ab/50/4f594db59dd034cb4fc9a6b5aae6986c5749bdd12e71eb12f0
Successfully built diff_gaussian_rasterization


In [ ]:
!pip install --no-build-isolation ./fused-ssim 2>&1 | tail -5

  Created wheel for fused_ssim: filename=fused_ssim-0.0.0-cp313-cp313-linux_x86_64.whl size=3133740 sha256=a81d1da98bae25d2466ba177aae5fc54a6018919717f87ecbf42ef725245c947
  Stored in directory: /tmp/pip-ephem-wheel-cache-xgkvjifp/wheels/19/65/37/f3c244855d9cef192117b25ebabc4ca2981be5e1b18a423d77
Successfully built fused_ssim


In [ ]:
import torch
from diff_gaussian_rasterization import GaussianRasterizer
from simple_knn._C import distCUDA2
from fused_ssim import fused_ssim
print("All OK")

All OK


In [ ]:
%cd /content/gaussian-splatting
!wget -q --show-progress https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip
!unzip -q tandt_db.zip
!ls tandt db
!ls tandt/truck
!ls tandt/truck/images | wc -l

/content/gaussian-splatting
tandt_db.zip        100%[===================>] 651.00M  16.6MB/s    in 39s     
db:
drjohnson  playroom

tandt:
train  truck
images	sparse
251


In [ ]:
%cd /content/gaussian-splatting
!python train.py -s tandt/truck -m /content/drive/MyDrive/gs_lab/truck --eval

/content/gaussian-splatting
I0000 00:00:1793357655.943715    9434 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793357656.015520    9434 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1793357658.255703    9434 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793357658.256343    9434 cudart_stub.cc:31] Could not find cuda drivers on your ma

In [ ]:
!cp /content/gaussian-splatting/tandt_db.zip /content/drive/MyDrive/gs_lab/
!ls -lh /content/drive/MyDrive/gs_lab/truck/point_cloud/*/

/content/drive/MyDrive/gs_lab/truck/point_cloud/iteration_30000/:
total 485M
-rw------- 1 root root 485M Oct 30 11:37 point_cloud.ply

/content/drive/MyDrive/gs_lab/truck/point_cloud/iteration_7000/:
total 399M
-rw------- 1 root root 399M Oct 30 11:01 point_cloud.ply


In [ ]:
!ls -lh /content/drive/MyDrive/gs_lab/

total 652M
-rw------- 1 root root 652M Oct 30 11:37 tandt_db.zip
drwx------ 3 root root 4.0K Oct 30 11:37 truck
